In [2]:
import sys
!{sys.executable} -m pip install polars pandas numpy matplotlib seaborn scikit-learn pyarrow psutil

   ---------------------------------------- 0.0/833.4 kB ? eta -:--:--
   ---------------------------------------- 0.0/833.4 kB ? eta -:--:--
   ---------------------------------------- 0.0/833.4 kB ? eta -:--:--
   ---------------------------------------- 0.0/833.4 kB ? eta -:--:--
   ------------ --------------------------- 262.1/833.4 kB ? eta -:--:--
   ------------------------------------- -- 786.4/833.4 kB 2.5 MB/s eta 0:00:01
   ---------------------------------------- 833.4/833.4 kB 1.9 MB/s  0:00:01
   ---------------------------------------- 0.0/52.0 MB ? eta -:--:--
   - -------------------------------------- 1.3/52.0 MB 5.3 MB/s eta 0:00:10
   - -------------------------------------- 1.6/52.0 MB 4.9 MB/s eta 0:00:11
   - -------------------------------------- 2.4/52.0 MB 3.8 MB/s eta 0:00:14
   -- ------------------------------------- 3.1/52.0 MB 3.5 MB/s eta 0:00:14
   -- ------------------------------------- 3.7/52.0 MB 3.4 MB/s eta 0:00:15
   ---- -----------------------

In [6]:
import polars as pl
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

pl.Config.set_tbl_rows(20)
pl.Config.set_tbl_cols(15)

UMBRAL_RETRASO = 15  # minutos: vuelo "retrasado" si ARRIVAL_DELAY > 15

RUTA_DATOS = Path(r"E:\Ing ciencia de datos\Septimo cuatri\Computacion paralela\clase 5\Tarea_3\data\raw")
print("RUTA_DATOS:", RUTA_DATOS)

RUTA_DATOS: E:\Ing ciencia de datos\Septimo cuatri\Computacion paralela\clase 5\Tarea_3\data\raw


In [7]:
df = pl.read_csv(
    str(RUTA_DATOS / "flights.csv"),
    null_values=["", "NA"],
    infer_schema_length=10000,
)
print(f"Dimensiones: {df.height:,} filas x {df.width} columnas")

Dimensiones: 5,819,079 filas x 31 columnas


In [8]:
print("--- Esquema (tipos de datos) ---")
for nombre, tipo in df.schema.items():
    print(f"  {nombre:<22} {tipo}")

df.head(5)

--- Esquema (tipos de datos) ---
  YEAR                   Int64
  MONTH                  Int64
  DAY                    Int64
  DAY_OF_WEEK            Int64
  AIRLINE                String
  FLIGHT_NUMBER          Int64
  TAIL_NUMBER            String
  ORIGIN_AIRPORT         String
  DESTINATION_AIRPORT    String
  SCHEDULED_DEPARTURE    Int64
  DEPARTURE_TIME         Int64
  DEPARTURE_DELAY        Int64
  TAXI_OUT               Int64
  WHEELS_OFF             Int64
  SCHEDULED_TIME         Int64
  ELAPSED_TIME           Int64
  AIR_TIME               Int64
  DISTANCE               Int64
  WHEELS_ON              Int64
  TAXI_IN                Int64
  SCHEDULED_ARRIVAL      Int64
  ARRIVAL_TIME           Int64
  ARRIVAL_DELAY          Int64
  DIVERTED               Int64
  CANCELLED              Int64
  CANCELLATION_REASON    String
  AIR_SYSTEM_DELAY       Int64
  SECURITY_DELAY         Int64
  AIRLINE_DELAY          Int64
  LATE_AIRCRAFT_DELAY    Int64
  WEATHER_DELAY          Int64


YEAR,MONTH,DAY,DAY_OF_WEEK,AIRLINE,FLIGHT_NUMBER,TAIL_NUMBER,…,CANCELLED,CANCELLATION_REASON,AIR_SYSTEM_DELAY,SECURITY_DELAY,AIRLINE_DELAY,LATE_AIRCRAFT_DELAY,WEATHER_DELAY
i64,i64,i64,i64,str,i64,str,…,i64,str,i64,i64,i64,i64,i64
2015,1,1,4,"""AS""",98,"""N407AS""",…,0,null,null,null,null,null,null
2015,1,1,4,"""AA""",2336,"""N3KUAA""",…,0,null,null,null,null,null,null
2015,1,1,4,"""US""",840,"""N171US""",…,0,null,null,null,null,null,null
2015,1,1,4,"""AA""",258,"""N3HYAA""",…,0,null,null,null,null,null,null
2015,1,1,4,"""AS""",135,"""N527AS""",…,0,null,null,null,null,null,null


In [9]:
# RETRASADO: 1 si el retraso de llegada supera el umbral, 0 si no.
# Queda nulo en vuelos cancelados/desviados (sin ARRIVAL_DELAY).
df = df.with_columns(
    (pl.col("ARRIVAL_DELAY") > UMBRAL_RETRASO).cast(pl.Int8).alias("RETRASADO")
)

dist_objetivo = (
    df.group_by("RETRASADO")
      .agg(pl.len().alias("conteo"))
      .with_columns((pl.col("conteo") / df.height * 100).round(2).alias("porcentaje"))
      .sort("RETRASADO")
)
print("Distribución de la variable objetivo:")
dist_objetivo

Distribución de la variable objetivo:


RETRASADO,conteo,porcentaje
i8,u32,f64
null,105071,1.81
0,4690510,80.61
1,1023498,17.59


In [10]:
df.describe()

statistic,YEAR,MONTH,DAY,DAY_OF_WEEK,AIRLINE,FLIGHT_NUMBER,…,CANCELLATION_REASON,AIR_SYSTEM_DELAY,SECURITY_DELAY,AIRLINE_DELAY,LATE_AIRCRAFT_DELAY,WEATHER_DELAY,RETRASADO
str,f64,f64,f64,f64,str,f64,…,str,f64,f64,f64,f64,f64,f64
"""count""",5.819079e6,5.819079e6,5.819079e6,5.819079e6,"""5819079""",5.819079e6,…,"""89884""",1.063439e6,1.063439e6,1.063439e6,1.063439e6,1.063439e6,5.714008e6
"""null_count""",0.0,0.0,0.0,0.0,"""0""",0.0,…,"""5729195""",4.75564e6,4.75564e6,4.75564e6,4.75564e6,4.75564e6,105071.0
"""mean""",2015.0,6.524085,15.704594,3.926941,null,2173.092742,…,null,13.480568,0.076154,18.969547,23.472838,2.91529,0.179121
"""std""",0.0,3.405137,8.783425,1.988845,null,1757.063999,…,null,28.003679,2.14346,48.161642,43.197018,20.433336,0.383454
"""min""",2015.0,1.0,1.0,1.0,"""AA""",1.0,…,"""A""",0.0,0.0,0.0,0.0,0.0,0.0
"""25%""",2015.0,4.0,8.0,2.0,null,730.0,…,null,0.0,0.0,0.0,0.0,0.0,0.0
"""50%""",2015.0,7.0,16.0,4.0,null,1690.0,…,null,2.0,0.0,2.0,3.0,0.0,0.0
"""75%""",2015.0,9.0,23.0,6.0,null,3230.0,…,null,18.0,0.0,19.0,29.0,0.0,0.0
"""max""",2015.0,12.0,31.0,7.0,"""WN""",9855.0,…,"""D""",1134.0,573.0,1971.0,1331.0,1211.0,1.0


In [11]:
nulos = (
    df.null_count()
      .transpose(include_header=True, header_name="columna", column_names=["n_nulos"])
      .with_columns((pl.col("n_nulos") / df.height * 100).round(2).alias("pct_nulos"))
      .filter(pl.col("n_nulos") > 0)
      .sort("n_nulos", descending=True)
)
print("Columnas con valores faltantes:")
nulos

Columnas con valores faltantes:


columna,n_nulos,pct_nulos
str,u32,f64
"""CANCELLATION_REASON""",5729195,98.46
"""AIR_SYSTEM_DELAY""",4755640,81.72
"""SECURITY_DELAY""",4755640,81.72
"""AIRLINE_DELAY""",4755640,81.72
"""LATE_AIRCRAFT_DELAY""",4755640,81.72
"""WEATHER_DELAY""",4755640,81.72
"""ELAPSED_TIME""",105071,1.81
"""AIR_TIME""",105071,1.81
"""ARRIVAL_DELAY""",105071,1.81
